## Validation with Berlin data

In [57]:
import pandas as pd
import geopandas as gpd

In [58]:
import os
from sqlalchemy import create_engine,text
db_url = (
    f"postgresql://{os.getenv('POSTGRES_USER')}:"
    f"{os.getenv('POSTGRES_PASSWORD')}@"
    f"localhost:5432/"
    f"{os.getenv('POSTGRES_DB')}"
)

engine = create_engine(db_url)


In [63]:
line_stations_gdf = gpd.GeoDataFrame.from_file("berlin_stations_line_ref.geojson")
line_stations_gdf.to_postgis("counting_stations_lines_berlin",engine, if_exists="replace")

In [64]:
point_stations_gdf = gpd.GeoDataFrame.from_file("berlin_stations_ref_clean.geojson")
count_stations_gdf = gpd.GeoDataFrame.from_file("berlin_stations_count.csv")
count_stations_gdf["date"] = pd.to_datetime(count_stations_gdf["time"]).dt.date
count_stations_gdf["count"] = count_stations_gdf["count"].replace('','0').astype(float)
daily_df = count_stations_gdf[["id","date","count"]].groupby(["id","date"],as_index=False).sum()
num_days_df = daily_df[["id","date"]].groupby(["id"],as_index=False).count().rename(columns={"date":"num_days"})
total_count_df = daily_df[["id","count"]].groupby(["id"],as_index=False).sum()
average_daily_df = pd.merge(total_count_df,num_days_df)
average_daily_df["count_aadt"] = average_daily_df["count"] / average_daily_df["num_days"]
final_merged_gdf = point_stations_gdf[["id","geometry"]].merge(average_daily_df,how="inner")
final_merged_gdf.to_postgis("counting_stations_points_berlin",engine, if_exists="replace")

In [65]:
with engine.begin() as conn:
    conn.execute(text("""
    ALTER TABLE counting_stations_lines_berlin
    ADD COLUMN IF NOT EXISTS count_stations INTEGER,
    ADD COLUMN IF NOT EXISTS num_points INTEGER;"""))

    conn.execute(text("""WITH results AS (
    SELECT l.id as id, l.geometry as geom, SUM(count_aadt) as count_stations, count(*) as num_points FROM
    (counting_stations_lines_berlin l
    INNER JOIN counting_stations_points_berlin p
    ON ST_Distance(l.geometry::geography,p.geometry::geography) < 100)
    GROUP BY l.id,l.geometry
    )
    UPDATE counting_stations_lines_berlin s
    SET num_points = results.num_points,count_stations = results.count_stations
    FROM results
    WHERE s.id = results.id;
    """))



In [67]:
with engine.begin() as conn:
    conn.execute(text("""
    ALTER TABLE counting_stations_lines_berlin
    DROP COLUMN IF EXISTS count_base;"""))

    conn.execute(text("""
    ALTER TABLE counting_stations_lines_berlin
    ADD COLUMN count_base INTEGER;

    WITH counts_base AS (
        SELECT g.id as id, sum(l.count_occ) as count_base
        FROM counting_stations_lines_berlin g
        INNER JOIN
        (SELECT * FROM result_berlin_base) l
        ON ST_Intersects(g.geometry, l.geom)
        GROUP BY g.id
    )
    UPDATE counting_stations_lines_berlin h
    SET count_base = counts_base.count_base
    FROM counts_base
    WHERE h.id = counts_base.id ;

    """))

    conn.execute(text("""
    ALTER TABLE counting_stations_lines_berlin
    DROP COLUMN IF EXISTS count_sim;"""))

    conn.execute(text("""
    ALTER TABLE counting_stations_lines_berlin
    ADD COLUMN count_sim INTEGER;

    WITH counts_sim AS (
        SELECT g.id as id, sum(l.count_occ) as count_sim
        FROM counting_stations_lines_berlin g
        INNER JOIN
        (SELECT * FROM final_table WHERE fetched_city = 'Berlin') l
        ON ST_Intersects(g.geometry, l.geom)
        GROUP BY g.id
    )
    UPDATE counting_stations_lines_berlin h
    SET count_sim = counts_sim.count_sim
    FROM counts_sim
    WHERE h.id = counts_sim.id ;

"""))


In [68]:
query = """SELECT * FROM counting_stations_lines_berlin"""

berlin_gdf = gpd.GeoDataFrame.from_postgis(query, engine, geom_col="geometry")
berlin_gdf

,id,geometry,count_stations,num_points,count_base,count_sim
0,1,"MULTILINESTRING ((13.33307 52.48813, 13.33319 ...",1673,1,62,65
1,2,"MULTILINESTRING ((13.32671 52.51324, 13.32682 ...",4216,2,4118,1956
2,3,"MULTILINESTRING ((13.36979 52.48815, 13.36978 ...",3272,1,242,687
3,4,"MULTILINESTRING ((13.37331 52.49229, 13.37324 ...",5464,2,6992,4388
4,5,"MULTILINESTRING ((13.3724 52.52753, 13.37254 5...",4509,2,3893,2833
5,6,"MULTILINESTRING ((13.35218 52.53964, 13.35222 ...",1240,1,34,93
6,7,"MULTILINESTRING ((13.41233 52.53154, 13.41244 ...",2886,1,1488,874
7,8,"MULTILINESTRING ((13.41205 52.54304, 13.41215 ...",2083,1,1733,1710
8,9,"MULTILINESTRING ((13.41749 52.52184, 13.41743 ...",2896,1,1645,1465
9,10,"MULTILINESTRING ((13.42617 52.51901, 13.42614 ...",2310,1,1258,689


In [69]:
berlin_gdf["rank_real"] = berlin_gdf["count_stations"].rank(ascending=False)
berlin_gdf["rank_base"] = berlin_gdf["count_base"].rank(ascending=False)
berlin_gdf["rank_sim"] = berlin_gdf["count_sim"].rank(ascending=False)
berlin_gdf["rank_diff"] = berlin_gdf["rank_real"] - berlin_gdf["rank_sim"]

berlin_gdf

,id,geometry,count_stations,num_points,count_base,count_sim,rank_real,rank_base,rank_sim,rank_diff
0,1,"MULTILINESTRING ((13.33307 52.48813, 13.33319 ...",1673,1,62,65,16.0,16.0,18.0,-2.0
1,2,"MULTILINESTRING ((13.32671 52.51324, 13.32682 ...",4216,2,4118,1956,8.0,3.0,5.0,3.0
2,3,"MULTILINESTRING ((13.36979 52.48815, 13.36978 ...",3272,1,242,687,9.0,15.0,13.0,-4.0
3,4,"MULTILINESTRING ((13.37331 52.49229, 13.37324 ...",5464,2,6992,4388,3.0,2.0,2.0,1.0
4,5,"MULTILINESTRING ((13.3724 52.52753, 13.37254 5...",4509,2,3893,2833,7.0,4.0,3.0,4.0
5,6,"MULTILINESTRING ((13.35218 52.53964, 13.35222 ...",1240,1,34,93,18.0,18.0,17.0,1.0
6,7,"MULTILINESTRING ((13.41233 52.53154, 13.41244 ...",2886,1,1488,874,11.0,10.0,10.0,1.0
7,8,"MULTILINESTRING ((13.41205 52.54304, 13.41215 ...",2083,1,1733,1710,15.0,8.0,7.0,8.0
8,9,"MULTILINESTRING ((13.41749 52.52184, 13.41743 ...",2896,1,1645,1465,10.0,9.0,9.0,1.0
9,10,"MULTILINESTRING ((13.42617 52.51901, 13.42614 ...",2310,1,1258,689,14.0,11.0,12.0,2.0


In [56]:
berlin_gdf["count_stations"].mean()

np.float64(3798.6666666666665)

In [73]:
from scipy.stats import spearmanr
from scipy.stats import pearsonr

res = spearmanr(berlin_gdf["rank_real"], berlin_gdf["rank_base"])

print("Spearman correlation:", res.statistic)
print("p-value:", res.pvalue)

Spearman correlation: 0.6821465428276574
p-value: 0.0018170810177521768


In [82]:
from scipy.stats import kendalltau

res = kendalltau(berlin_gdf["rank_real"], berlin_gdf["rank_base"])

print("kendalltau correlation:", res.statistic)
print("p-value:", res.pvalue)

kendalltau correlation: 0.4901960784313726
p-value: 0.003934978451529695


In [74]:
from scipy.stats import spearmanr
from scipy.stats import pearsonr

res = spearmanr(berlin_gdf["rank_real"], berlin_gdf["rank_sim"])

print("Spearman correlation:", res.statistic)
print("p-value:", res.pvalue)

Spearman correlation: 0.6904024767801857
p-value: 0.0015160249841227739


In [83]:
from scipy.stats import kendalltau

res = kendalltau(berlin_gdf["rank_real"], berlin_gdf["rank_sim"])

print("kendalltau correlation:", res.statistic)
print("p-value:", res.pvalue)

kendalltau correlation: 0.5163398692810458
p-value: 0.002244139045296836


In [80]:
res = pearsonr(berlin_gdf["count_stations"], berlin_gdf["count_base"])

print("Pearson correlation:", res.statistic)
print("p-value:", res.pvalue)

Pearson correlation: 0.7764243480897124
p-value: 0.00015135530147131237


In [81]:
res = pearsonr(berlin_gdf["count_stations"], berlin_gdf["count_sim"])

print("Pearson correlation:", res.statistic)
print("p-value:", res.pvalue)

Pearson correlation: 0.8484607827482284
p-value: 8.593559620949369e-06
